# v2 A × RootAnchored24 · 배포 판정 및 탐색적 T4 짝비교
고정된 세 클립에서 공식 A와 래퍼의 원소별 일치를 검증한다. exploratory는 배포 재현 실패를 기록하고 후보 strict load를 통과해야 성능을 측정한다. 학교 계정과 복구한 입력 경로를 사용한다.

In [ ]:
from pathlib import Path
import os, sys, subprocess, tempfile, json, hashlib, shutil, zipfile, datetime
from google.colab import drive, auth
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload
import torch
assert torch.cuda.is_available() and 'T4' in torch.cuda.get_device_name(), 'Tesla T4 required'
auth.authenticate_user()
credentials, _ = google.auth.default()
api = build('drive','v3',credentials=credentials,cache_discovery=False)
assert api.about().get(fields='user(emailAddress)').execute()['user']['emailAddress']=='darkk0729@ajou.ac.kr'
drive.mount('/content/drive')
root = Path(tempfile.mkdtemp(prefix='rewind-dualcamera-',dir='/content'))
repo = root/'repo'
sha = 'af4507d92bb3e6a57dbdbd44dae9c0b85c75ae63'
subprocess.run(['git','clone','--single-branch','--branch','integration/v2a-paired-t4-20261122','https://github.com/darkkyoung/REWIND-like-prototype.git',str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'checkout','--detach',sha],check=True)
assert subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip()==sha
print(root,torch.cuda.get_device_name(),torch.__version__)

학교 Drive의 공유 파일은 My Drive에 바로가기를 둔 뒤 아래 경로에 지정한다. 이미 복구한 portable bundle 디렉터리와 anchored.pt 경로도 지원한다. 공식 로더가 선택 멤버를 검증한다. 55 GB 전체 ZIP을 로컬에 복사하는 셀은 없다.

In [ ]:
bundle_path = ''  # @param {type:"string"}
anchored_path = ''  # @param {type:"string"}
mode = 'exploratory'  # @param ['exploratory','strict']
assert bundle_path and anchored_path, 'Enter actual recovered bundle directory or A ZIP and candidate PT or ZIP paths'
bundle = Path(bundle_path)
anchored = Path(anchored_path)
ids = [('1WiVnkAr6Q9i7eiGLzTDL1qarxCULSRXo','4591393d3aceb4bb4af9b387c4fda038bdcdd5fd5d613fdcc95f7cb50cdc736a'),
       ('1TgEUnqwBVjx4G4jY2Lwubpk5MCcW3iQb','51704fc55a7447ec3da17a36b5b94f7d584306fb1c9131b14fbfdabf4712b3a3')]
for fid, pin in ids:
    meta=api.files().get(fileId=fid,fields='name,size,sha256Checksum',supportsAllDrives=True).execute()
    assert meta['sha256Checksum']==pin
    print(meta['name'],meta['size'])
assert bundle.exists() and anchored.exists(), 'Enter the mounted package paths above'
runtime = root/'runtime'
result = root/'result'
logs = root/'logs'
logs.mkdir()
(logs/'identity.json').write_text(json.dumps({'mode':mode,'adapter_commit':sha,'inputs':ids,'torch':str(torch.__version__),'device':torch.cuda.get_device_name()}))
env={**os.environ,'CUBLAS_WORKSPACE_CONFIG':':4096:8','PYTHONPATH':os.pathsep.join(map(str,[runtime/'source',runtime/'tools',repo]))}
def run(name,args):
    with (logs/(name+'.log')).open('w') as f:
        status = subprocess.run([sys.executable,'-u',*map(str,args)],cwd=root,env=env,stdout=f,stderr=subprocess.STDOUT)
    print((logs/(name+'.log')).read_text()[-2500:])
    status.check_returncode()
run('prepare',[repo/'experiments/v2a_dualcamera/prepare.py','--bundle',bundle,'--anchored',anchored,'--dest',runtime])
run('test',['-m','unittest','-v','experiments.v2a_dualcamera.test_adapter','experiments.v2a_dualcamera.test_paired'])
run('camera-regression',['-m','unittest','discover','-v','-s',runtime/'source/tests','-p','test_v15_camera_training.py'])
run('split',['-m','experiments.v2a_dualcamera.audit_split'])

In [ ]:
run('gpu',['-m','experiments.v2a_dualcamera.run','--mode',mode,'--bundle',bundle,'--anchored',runtime/'anchored.pt','--work',root/'restore','--output',result])
print((result/'verdict.json').read_text())
summary=json.loads((result/'results.json').read_text())
for scenario,x in summary['clips'].items():
    print(scenario,{arm:{k:x[arm][k] for k in ['body_mpjpe_mm','rr_jpe_mm','hand_global_mpjpe_mm','whole_mpjpe_mm']} for arm in ['baseline','anchored']})

### 결과 또는 중단 증거 보관
GPU 비교가 중단되어도 이 셀은 실행할 수 있다. 원형 parity에 실패한 실행은 결과 ZIP의 상태가 incomplete로 표시된다.

In [ ]:
from io import BytesIO
verdict=json.loads((result/'verdict.json').read_text()) if (result/'verdict.json').exists() else {}
status='completed' if verdict.get('run_status')=='completed' else 'incomplete'
name='PairedT4_'+mode+'_'+status+'_'+datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')+'.zip'
archive=root/name
with zipfile.ZipFile(archive,'x',compression=zipfile.ZIP_DEFLATED) as z:
    for folder in (logs,result):
        if folder.exists():
            for p in sorted(folder.rglob('*')):
                if p.is_file(): z.write(p,p.relative_to(root))
with archive.open('rb') as f: expected=hashlib.file_digest(f,'sha256').hexdigest()
assert api.about().get(fields='user(emailAddress)').execute()['user']['emailAddress']=='darkk0729@ajou.ac.kr'
mark=api.files().create(body={'name':name},media_body=MediaFileUpload(str(archive),resumable=True),fields='id,webViewLink').execute()
buf=BytesIO()
loader=MediaIoBaseDownload(buf,api.files().get_media(fileId=mark['id']))
done=False
while not done: _,done=loader.next_chunk()
assert hashlib.sha256(buf.getbuffer()).hexdigest()==expected
print(status,expected,mark)